# Developer Flow & Interruption Data Merging



## Step 1: Import libraries and load the cleaned files

In [31]:
import pandas as pd
import numpy as np

In [32]:
fact_log = pd.read_csv("fact_developer_activity_log_cleaned.csv")

In [33]:
dim_developer = pd.read_csv("dim_developer_cleaned.csv")

In [34]:
dim_activity = pd.read_csv("dim_activity_type_cleaned.csv")

In [35]:
dim_interruption = pd.read_csv("dim_interruption_cleaned.csv")

In [36]:
dim_date = pd.read_csv("dim_date_cleaned.csv")

In [37]:
print("fact_log:", fact_log.shape)
print("dim_developer:", dim_developer.shape)
print("dim_activity:", dim_activity.shape)
print("dim_interruption:", dim_interruption.shape)
print("dim_date:", dim_date.shape)

fact_log: (158152, 11)
dim_developer: (150, 6)
dim_activity: (6, 4)
dim_interruption: (8, 6)
dim_date: (90, 10)


## Step 2: Double check the IDs still match up



In [38]:
print("developer_id in fact_log but missing from dim_developer:",
      (~fact_log["developer_id"].isin(dim_developer["developer_id"])).sum())
print("activity_id in fact_log but missing from dim_activity:",
      (~fact_log["activity_id"].isin(dim_activity["activity_id"])).sum())
print("interruption_id in fact_log but missing from dim_interruption:",
      (~fact_log["interruption_id"].isin(dim_interruption["interruption_id"])).sum())
print("date_key in fact_log but missing from dim_date:",
      (~fact_log["date_key"].isin(dim_date["date_key"])).sum())

developer_id in fact_log but missing from dim_developer: 0
activity_id in fact_log but missing from dim_activity: 0
interruption_id in fact_log but missing from dim_interruption: 0
date_key in fact_log but missing from dim_date: 0


## Step 3: Join everything together



In [39]:
merged = fact_log.merge(dim_developer, on="developer_id", how="left")
merged = merged.merge(dim_activity, on="activity_id", how="left")
merged = merged.merge(dim_interruption, on="interruption_id", how="left")
merged = merged.merge(dim_date, on="date_key", how="left")

print("Merged table shape:", merged.shape)

Merged table shape: (158152, 33)


In [40]:
print("Any missing values after the merge:", merged.isnull().sum().sum())

Any missing values after the merge: 229754


In [41]:
merged.head()

,log_id,developer_id,date_key,timestamp_start,timestamp_end,activity_id,interruption_id,session_duration_minutes,in_flow_state,context_switch_flag,cognitive_recovery_minutes,developer_name,team_name,seniority_level,primary_ide,timezone,activity_name,cognitive_category,is_deep_work,source_channel,interruption_category,urgency_tier,requires_action,avg_recovery_latency_min,date,day_of_week,day_number_in_week,is_weekend,week_number,month_name,quarter,year,sprint_name
0,1000001,101,20260101,2026-01-01 09:00:00,2026-01-01 09:39:00,1,6,39,True,1,7.24,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST),Active IDE Coding & Refactoring,Deep Work,True,Jira Notification Digest,Tooling & Automation,Noise/Spam,False,8.5,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01
1,1000002,101,20260101,2026-01-01 09:39:00,2026-01-01 10:06:00,1,4,27,True,1,30.40,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST),Active IDE Coding & Refactoring,Deep Work,True,PagerDuty Incident Alert,Production Alert,Critical,True,26.5,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01
2,1000003,101,20260101,2026-01-01 10:06:00,2026-01-01 10:44:00,1,0,38,True,0,0.00,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST),Active IDE Coding & Refactoring,Deep Work,True,None (Continuous Focus),NaN,NaN,False,0.0,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01
3,1000004,101,20260101,2026-01-01 10:44:00,2026-01-01 10:59:00,6,0,15,False,0,0.00,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST),Triaging Slack & Notifications,Friction / Interruption,False,None (Continuous Focus),NaN,NaN,False,0.0,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01
4,1000005,101,20260101,2026-01-01 10:59:00,2026-01-01 11:14:00,3,0,15,False,0,0.00,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST),Code Review & PR Inspection,Focused Task,False,None (Continuous Focus),NaN,NaN,False,0.0,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01


## Step 4: Take a quick look at the merged columns



In [42]:
merged.columns.tolist()

['log_id', 'developer_id', 'date_key', 'timestamp_start', 'timestamp_end', 'activity_id', 'interruption_id', 'session_duration_minutes', 'in_flow_state', 'context_switch_flag', 'cognitive_recovery_minutes', 'developer_name', 'team_name', 'seniority_level', 'primary_ide', 'timezone', 'activity_name', 'cognitive_category', 'is_deep_work', 'source_channel', 'interruption_category', 'urgency_tier', 'requires_action', 'avg_recovery_latency_min', 'date', 'day_of_week', 'day_number_in_week', 'is_weekend', 'week_number', 'month_name', 'quarter', 'year', 'sprint_name']

## Step 5: Save the merged file

In [43]:
merged.to_csv("activity_log_merged.csv", index=False)
print("Merged file saved! Shape:", merged.shape)

Merged file saved! Shape: (158152, 33)


## Summary

- Loaded the 6 cleaned files from the previous notebook
- Re-checked that all the IDs still matched up correctly before joining
- Merged the main activity log with all 4 lookup tables into one combined table
- Confirmed no missing values were introduced by the merge
- Saved the final merged file (`activity_log_merged.csv`), ready for analysis or Power BI
